# Lab 04.2 — Ozone tramite API S3

L'**S3 Gateway** espone i bucket del volume speciale **`/s3v`** con l'API Amazon S3.
Lo stesso dato è quindi accessibile in due modi:

* da Spark/Hadoop con `ofs://om/s3v/<bucket>/...`
* da qualunque client S3 (boto3, AWS CLI, PyArrow, ...) con `s3://<bucket>/...` sull'endpoint `http://s3g:9878`

In questo laboratorio la sicurezza è disattivata: qualunque access key è accettata.

Nota: i bucket creati via S3 hanno di default layout `OBJECT_STORE` (puro object store, non leggibile con `ofs://`).
Il laboratorio imposta `ozone.s3g.default.bucket.layout=FILE_SYSTEM_OPTIMIZED` (vedi `ozone.env`) per usarli da entrambe le parti.

In [ ]:
import os, boto3
from pyspark.sql import SparkSession, functions as F

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://s3g:9878")
s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT, region_name="us-east-1",
                  aws_access_key_id="lab", aws_secret_access_key="lab")

spark = (SparkSession.builder
         .appName("Lab04-Ozone-S3")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.hadoop.fs.ofs.impl", "org.apache.hadoop.fs.ozone.RootedOzoneFileSystem")
         .config("spark.hadoop.ozone.om.address", "om:9862")
         .config("spark.hadoop.ozone.replication", "1")
         .getOrCreate())

## 1. Creare un bucket e caricare oggetti con boto3

In [ ]:
BUCKET = "dati-s3"
if BUCKET not in [b["Name"] for b in s3.list_buckets()["Buckets"]]:
    s3.create_bucket(Bucket=BUCKET)
s3.upload_file("/home/jovyan/datasets/alice_in_wonderland.txt", BUCKET, "testi/alice.txt")

print("Bucket S3:", [b["Name"] for b in s3.list_buckets()["Buckets"]])
for o in s3.list_objects_v2(Bucket=BUCKET)["Contents"]:
    print(f"  {o['Key']:25s} {o['Size']:>10,d} byte")

## 2. Spark legge lo stesso oggetto da `ofs://om/s3v/...`

In [ ]:
testo = spark.read.text(f"ofs://om/s3v/{BUCKET}/testi/alice.txt")
print("Righe lette da Spark:", testo.count())

(testo.select(F.explode(F.split(F.lower("value"), "[^a-z]+")).alias("parola"))
      .where(F.length("parola") > 5)
      .groupBy("parola").count().orderBy(F.desc("count")).show(5))

## 3. Spark scrive Parquet, i client S3 lo leggono

In [ ]:
temperature = (spark.range(0, 24 * 365)
               .withColumn("ora", F.expr("timestamp_seconds(1704067200 + id * 3600)"))
               .withColumn("citta", F.element_at(F.array(F.lit("Roma"), F.lit("Milano"), F.lit("Napoli")),
                                                 (F.col("id") % 3 + 1).cast("int")))
               .withColumn("gradi", F.round(15 + 10 * F.sin(F.col("id") / 600) + F.randn(3) * 2, 1))
               .drop("id"))

temperature.write.mode("overwrite").parquet(f"ofs://om/s3v/{BUCKET}/meteo")

for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="meteo/")["Contents"]:
    print(f"  {o['Key']:75s} {o['Size']:>8,d} byte")

In [ ]:
# PyArrow legge la cartella Parquet direttamente via S3 (senza Spark)
import pyarrow.fs as pafs, pyarrow.parquet as pq

s3fs = pafs.S3FileSystem(endpoint_override=S3_ENDPOINT, scheme="http", region="us-east-1",
                         access_key="lab", secret_key="lab")
tabella = pq.read_table(f"{BUCKET}/meteo", filesystem=s3fs)
df = tabella.to_pandas()
print(tabella.num_rows, "righe lette con PyArrow via S3")
df.groupby("citta")["gradi"].agg(["min", "mean", "max"]).round(1)

## 4. Scaricare un oggetto con boto3

In [ ]:
nome = [o["Key"] for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="meteo/part-")["Contents"]][0]
s3.download_file(BUCKET, nome, "/tmp/meteo.parquet")
print(pq.read_metadata("/tmp/meteo.parquet"))

In [ ]:
spark.stop()